# Split grouped by gas provision tree and bounty report (A15)

The positives come from 311 bounty reports (Commonwealth discussions). Addresses in one report were found with
one rationale, often funding links. If a report's Sybils sit in different gas provision trees, the tree split
can place them on both sides, and test performance may partly reflect report-level similarity.

- **Part A, diagnostics (no training).** `report_leakage`, an analogue of `sp.leakage_report`: test Sybils
  whose report, GitHub issue or reporter also has a training Sybil, under the random split, the tree split
  (seed 42) and the tree split over the 10 seeds in `sp.SPLIT_SEEDS`.
- **Part B, union groups.** Rows joined if they share a gas provision tree or a bounty report (Commonwealth
  link, plus GitHub issue where present); `sp.make_splits` is reused unchanged by overwriting the group
  column on a copy. **Checks** (before any model is trained): each partition's Sybil rate within 0.3
  percentage points of 4.19 % and its size within 1 point of 49 / 21 / 30 (**stop** if either fails); no
  component holds more than 5 % of all Sybils (**warning**). The first run treated the 5 % rule as a stop and
  ended with `stopped_for_review`: one component holds 45 % of Sybils. On review it was made a warning, and the
  composition it causes is measured instead (below). The partitioner is not modified and no edges are dropped.
- **Composition.** The components over 5 % of Sybils (`BIG`) are placed whole, largest first, so they change
  which Sybils each partition holds. Sybils by category and in `BIG`, per partition, under both splits.
- **Part C**: XGBoost and LightGBM, hyperparameters from `02` (chosen on the tree split; not re-searched),
  3 seeds, on the union split.
- **Part D**: LightGBM on the 10 union splits against the tree split, both trained here (same machine). Also a
  **same-composition comparison**: both models scored only on their test rows outside `BIG`, so the two test
  sets are drawn from the same population.

**Results**: `results/15_split_tree_report.json`.

## Setup

In [1]:
import os, platform, time, warnings
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import f1_score, average_precision_score, roc_auc_score
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)
PLATFORM = dict(machine=platform.machine(), processor=platform.processor(), system=platform.system(),
                python=platform.python_version(), cpu_count=os.cpu_count(), n_jobs=sp.N_JOBS)
print('Platform:', PLATFORM)

DATA_DIR = './data'
FEATS = sp.FEATS
df, _, _ = sp.build_master_df(DATA_DIR, verbose=False)

fcfs = pd.read_csv(sp.data_paths(DATA_DIR)['sybil'])
fcfs.columns = fcfs.columns.str.lower().str.strip()
fcfs['address'] = fcfs['address'].str.lower()
assert fcfs['address'].is_unique
F = fcfs.set_index('address')
KEYS = {   # bounty-report keys per row of df; NaN where the address is not on the list or the field is empty
    'report': df['addr'].map(F['commonwealth report link']),
    'github_issue': df['addr'].map(F['github issue']),
    'reporter': df['addr'].map(F['reward address1']),
}
syb = df['sybil'] == 1
assert KEYS['report'][syb].notna().all() and KEYS['report'][~syb].isna().all()   # every Sybil matched, no non-Sybil
n_syb = int(syb.sum())
print(f"{n_syb:,} Sybils; {KEYS['report'][syb].nunique()} reports, "
      f"{KEYS['github_issue'][syb].nunique()} GitHub issues ({KEYS['github_issue'][syb].notna().mean():.1%} of Sybils), "
      f"{KEYS['reporter'][syb].nunique()} reporters")
top10 = KEYS['report'][syb].value_counts().head(10).sum() / n_syb
print(f'Ten largest reports: {top10:.1%} of Sybils; largest reporter: {KEYS["reporter"][syb].value_counts().iloc[0] / n_syb:.1%}')
gh_reports = KEYS['report'][syb].groupby(KEYS['github_issue'][syb]).nunique()
print(f'GitHub issues linked to more than one report: {(gh_reports > 1).sum()}')

Code commit: 9753ef1
Platform: {'machine': 'x86_64', 'processor': 'x86_64', 'system': 'Linux', 'python': '3.11.15', 'cpu_count': 4, 'n_jobs': 4}


18,211 Sybils; 311 reports, 144 GitHub issues (35.2% of Sybils), 145 reporters
Ten largest reports: 35.6% of Sybils; largest reporter: 8.5%
GitHub issues linked to more than one report: 0


## Part A. Report-level overlap between train and test (no training)

`report_leakage(df, S, key)`: test Sybils whose `key` group also contains a training Sybil (training
partition before upsampling), with the number of keys involved.

In [2]:
def report_leakage(df, S, key):
    """Test Sybils whose key group (report link, GitHub issue or reporter) also has a training Sybil."""
    syb = df['sybil'] == 1
    tr, te = S['idx_train'], S['idx_test']
    k_tr = set(key.loc[tr][syb.loc[tr]].dropna())
    k_te = key.loc[te][syb.loc[te]]
    shared = k_te.isin(k_tr)
    return dict(test_sybils=int(syb.loc[te].sum()), test_sybils_with_key=int(k_te.notna().sum()),
                test_sybils_sharing_key_with_train_sybil=int(shared.sum()),
                keys_in_test=int(k_te.nunique()), keys_shared_with_train=int(k_te[shared].nunique()))

def overlap_row(split, seed, S):
    row = dict(split=split, seed=seed,
               tree=sp.leakage_report(df, S, verbose=False)['test_sybils_sharing_tree_with_train_sybil'])
    for name, key in KEYS.items():
        r = report_leakage(df, S, key)
        row[f'{name}'] = r['test_sybils_sharing_key_with_train_sybil']
        row[f'{name}_keys_shared'] = r['keys_shared_with_train']
        row[f'{name}_keys_in_test'] = r['keys_in_test']
    row['test_sybils'] = r['test_sybils']
    return row

overlap = [overlap_row('random', sp.SEED, sp.make_splits(df, FEATS, method='random', seed=sp.SEED))]
overlap += [overlap_row('tree', s, sp.make_splits(df, FEATS, method='group', seed=s)) for s in sp.SPLIT_SEEDS]
overlap = pd.DataFrame(overlap)
cols = ['tree', 'report', 'github_issue', 'reporter']
print('Test Sybils sharing a key with a training Sybil (of', overlap['test_sybils'].iloc[0], 'test Sybils):')
print(overlap[['split', 'seed', 'test_sybils'] + cols + ['report_keys_shared', 'report_keys_in_test']].to_string(index=False))
tree10 = overlap[overlap['split'] == 'tree']
assert len(tree10) == 10 and tree10['seed'].is_unique
overlap_summary = {
    'random_seed42': overlap.iloc[0][cols + ['report_keys_shared', 'report_keys_in_test']].astype(int).to_dict(),
    'tree_seed42': tree10.iloc[0][cols + ['report_keys_shared', 'report_keys_in_test']].astype(int).to_dict(),
    'tree_10seeds': {f'{c}_{a}': float(getattr(tree10[c], a)()) for c in cols + ['report_keys_shared'] for a in ('mean', 'std')},
}
print('\nTree split, 10 seeds:', {k: round(v, 1) for k, v in overlap_summary['tree_10seeds'].items()})

Test Sybils sharing a key with a training Sybil (of 5463 test Sybils):
 split  seed  test_sybils  tree  report  github_issue  reporter  report_keys_shared  report_keys_in_test
random    42         5463   599    5434          1935      5461                 220                  242
  tree    42         5463     0    5406          1798      5445                 212                  239
  tree     1         5463     0    5354          1801      5417                 217                  251
  tree     2         5463     0    5423          1860      5456                 216                  243
  tree     3         5463     0    5389          1830      5457                 219                  243
  tree     4         5463     0    5411          1826      5442                 207                  237
  tree     5         5463     0    5410          1816      5445                 205                  237
  tree     6         5463     0    5420          1847      5457                 216      

## Part B. Union groups (gas provision tree ∪ bounty report)

Rows joined if they share a gas provision tree, a Commonwealth report link, or a GitHub issue; components of
the bipartite row–key graph. The optional stricter variant also joins rows that share a reporter
(`reward address1`).

In [3]:
def union_groups(df, fcfs, reporter=False):
    """Component id per row: rows joined if they share a provision tree or a bounty report.

    reporter=True also joins rows with the same bounty hunter (`reward address1`), the stricter variant.
    """
    F = fcfs.set_index('address')
    rep = df['addr'].map(F['commonwealth report link']).fillna('').astype(str)
    gh = df['addr'].map(F['github issue']).fillna('').astype(str)
    cols = [
        pd.Series('T:' + df['provision_tree'].astype(str), index=df.index),
        pd.Series(np.where(rep != '', 'R:' + rep, None), index=df.index),
        pd.Series(np.where(gh != '', 'G:' + gh, None), index=df.index),
    ]
    if reporter:
        ra = df['addr'].map(F['reward address1']).fillna('').astype(str)
        cols.append(pd.Series(np.where(ra != '', 'A:' + ra, None), index=df.index))
    keys = pd.concat(cols, axis=1)               # one column per grouping; None = no key
    # pandas >= 3 keeps missing values in stack(); dropna() is required, or None keys would
    # factorize to -1 and silently link unrelated rows
    long = keys.stack().dropna().rename('key').reset_index(level=1, drop=True)   # (row, key) pairs
    row_pos = df.index.get_indexer(long.index)
    key_id, key_uni = pd.factorize(long.values)
    assert (key_id >= 0).all() and (row_pos >= 0).all()
    n_rows, n_keys = len(df), len(key_uni)
    # bipartite graph rows <-> keys; components = union groups
    g = coo_matrix((np.ones(len(long)), (row_pos, n_rows + key_id)),
                   shape=(n_rows + n_keys, n_rows + n_keys))
    _, comp = connected_components(g, directed=False)
    return pd.Series(comp[:n_rows], index=df.index).astype(str)

TARGET = dict(Train=49.0, Validation=21.0, Test=30.0)
BASE_RATE = 100 * df['sybil'].mean()

def component_diagnostics(groups, label):
    """Component sizes, the stop conditions, and the split they produce (no model is trained)."""
    c = pd.DataFrame({'group': groups, 'sybil': df['sybil'], 'tree': df['provision_tree'],
                      'report': KEYS['report']}).groupby('group').agg(
        rows=('sybil', 'size'), sybils=('sybil', 'sum'), trees=('tree', 'nunique'), reports=('report', 'nunique'))
    c['sybil_share'] = c['sybils'] / n_syb
    top_rows = c.sort_values('rows', ascending=False).head(5)
    top_syb = c.sort_values('sybils', ascending=False).head(5)
    d = dict(variant=label, components=len(c), multi_row_components=int((c['rows'] > 1).sum()),
             largest_by_rows=top_rows.reset_index(drop=True).to_dict('records'),
             largest_by_sybils=top_syb.reset_index(drop=True).to_dict('records'),
             max_component_sybil_share=float(c['sybil_share'].max()))
    df_u = df.copy()
    df_u['provision_tree'] = groups                    # make_splits groups by this column
    S_u = sp.make_splits(df_u, FEATS, method='group', seed=sp.SEED)   # its assert now guards the union groups
    summ = sp.split_summary(df, S_u)
    summ['SizePct'] = 100 * summ['Total'] / len(df)
    d['split_summary'] = summ.to_dict('records')
    d['big_components'] = c.index[c['sybil_share'] > 0.05].tolist()   # components over 5 % of Sybils
    warnings_ = {'max component Sybil share <= 5%': d['max_component_sybil_share'] <= 0.05}
    checks = {}
    for r in summ.itertuples():
        checks[f'{r.Split} Sybil rate within 0.3 pp of {BASE_RATE:.2f}%'] = bool(abs(r.SybilPct - BASE_RATE) <= 0.3)
        checks[f'{r.Split} size within 1 pt of {TARGET[r.Split]:.0f}%'] = bool(abs(r.SizePct - TARGET[r.Split]) <= 1.0)
    d['checks'], d['warnings'] = checks, warnings_
    d['passed'] = all(checks.values())
    d['warned'] = not all(warnings_.values())
    # Overlap under the union split, on the original trees (df) and on the report keys
    d['leakage_original_trees'] = sp.leakage_report(df, S_u, verbose=False)
    d['report_leakage'] = {k: report_leakage(df, S_u, v) for k, v in KEYS.items()}
    print(f"=== {label}: {d['components']:,} components ({d['multi_row_components']:,} with more than one row)")
    print('Largest by rows:'); print(top_rows.round(4).to_string())
    print('Largest by Sybils:'); print(top_syb.round(4).to_string())
    print(summ.round(2).to_string(index=False))
    for k, ok in checks.items():
        print(f"  {'PASS' if ok else 'FAIL'}  {k}")
    for k, ok in warnings_.items():
        print(f"  {'PASS' if ok else 'WARN'}  {k}")
    print(f"  Test Sybils sharing an original tree with a train Sybil: "
          f"{d['leakage_original_trees']['test_sybils_sharing_tree_with_train_sybil']}; sharing a report: "
          f"{d['report_leakage']['report']['test_sybils_sharing_key_with_train_sybil']}; a GitHub issue: "
          f"{d['report_leakage']['github_issue']['test_sybils_sharing_key_with_train_sybil']}\n")
    return d, df_u, S_u

diag, df_u, S_u = component_diagnostics(union_groups(df, fcfs), 'tree + report + GitHub issue')
# The union split cannot leave report-level overlap (asserted whatever the stop decision)
assert diag['leakage_original_trees']['test_sybils_sharing_tree_with_train_sybil'] == 0
assert diag['report_leakage']['report']['test_sybils_sharing_key_with_train_sybil'] == 0
assert diag['report_leakage']['github_issue']['test_sybils_sharing_key_with_train_sybil'] == 0

=== tree + report + GitHub issue: 355,619 components (22,649 with more than one row)
Largest by rows:
        rows  sybils  trees  reports  sybil_share
group                                            
4      13346    8260   6651       57       0.4536
1175     998       0      1        0       0.0000
757      954     954    954        1       0.0524
227      843     843    843        1       0.0463
214      733       2      2        1       0.0001
Largest by Sybils:
        rows  sybils  trees  reports  sybil_share
group                                            
4      13346    8260   6651       57       0.4536
757      954     954    954        1       0.0524
227      843     843    843        1       0.0463
716      529     529    529        1       0.0290
528      500     500    500        1       0.0275
     Split  Total  Sybil  NonSybil  SybilPct  SizePct
     Train 213046   8924    204122      4.19     49.0
Validation  91305   3824     87481      4.19     21.0
      Test 130435

### Optional stricter variant: also group by reporter

Run only if it passes the same stop conditions; its diagnostics are reported either way.

In [4]:
diag_reporter, df_ur, S_ur = component_diagnostics(union_groups(df, fcfs, reporter=True),
                                                   'tree + report + GitHub issue + reporter')

=== tree + report + GitHub issue + reporter: 355,461 components (22,530 with more than one row)
Largest by rows:
        rows  sybils  trees  reports  sybil_share
group                                            
4      16840   10071   8394      153       0.5530
225     1552    1549   1549        9       0.0851
525     1029    1029   1029        2       0.0565
1167     998       0      1        0       0.0000
750      983     983    983        2       0.0540
Largest by Sybils:
        rows  sybils  trees  reports  sybil_share
group                                            
4      16840   10071   8394      153       0.5530
225     1552    1549   1549        9       0.0851
525     1029    1029   1029        2       0.0565
750      983     983    983        2       0.0540
1203     516     516    516        7       0.0283
     Split  Total  Sybil  NonSybil  SybilPct  SizePct
     Train 214192  10071    204121      4.70    49.26
Validation  90829   3348     87481      3.69    20.89
      

### Stop decision

In [5]:
STATUS = ('stopped_for_review' if not diag['passed'] else 'ok_with_warning' if diag['warned'] else 'ok')
failed = [k for k, ok in diag['checks'].items() if not ok]
warned = [k for k, ok in diag['warnings'].items() if not ok]
print('Status:', STATUS)
if failed:
    print('Failed conditions:', failed)
    print('Parts C and D are not run. The partitioner is unchanged and no edges were dropped.')
if warned:
    print('Warnings:', warned, '- Parts C and D run; the composition below shows what the large components change.')
RUN = STATUS != 'stopped_for_review'
interpretation_inputs = dict(
    status=STATUS, failed_conditions=failed, warnings=warned,
    report_overlap_random_seed42=overlap_summary['random_seed42']['report'],
    report_overlap_tree_seed42=overlap_summary['tree_seed42']['report'],
    report_overlap_tree_10seeds_mean=overlap_summary['tree_10seeds']['report_mean'],
    report_overlap_tree_10seeds_sd=overlap_summary['tree_10seeds']['report_std'],
    test_sybils=int(overlap['test_sybils'].iloc[0]),
    max_component_sybil_share=diag['max_component_sybil_share'],
    reporter_variant_passed=diag_reporter['passed'],
    reporter_variant_max_component_sybil_share=diag_reporter['max_component_sybil_share'],
)

Status: ok_with_warning
Warnings: ['max component Sybil share <= 5%'] - Parts C and D run; the composition below shows what the large components change.


### Composition: where the large components go

`BIG`: the union components holding more than 5 % of all Sybils. Per partition, under the tree split and the union
split (seed 42, and where `BIG` lands for each of the 10 seeds).

In [6]:
groups_u = df_u['provision_tree']
BIG = set(diag['big_components'])
in_big = groups_u.isin(BIG)
print(f"BIG: {len(BIG)} components, {int(in_big.sum()):,} rows, {int((in_big & syb).sum()):,} Sybils "
      f"({(in_big & syb).sum() / n_syb:.1%} of Sybils)")

def composition(S, label):
    rows = []
    for part, key in [('Train', 'idx_train'), ('Validation', 'idx_val'), ('Test', 'idx_test')]:
        idx = S[key]
        sy = df.loc[idx, 'sybil'] == 1
        cat = df.loc[idx, 'category'][sy].value_counts()
        rows.append(dict(split=label, partition=part, rows=len(idx), sybils=int(sy.sum()),
                         sybils_in_BIG=int((in_big.loc[idx] & sy).sum()),
                         reports=int(KEYS['report'].loc[idx][sy].nunique()),
                         **{f'sybils_{c}': int(cat.get(c, 0)) for c in ['IxL', 'IxI', 'IxE', 'No provider']}))
    return rows

S_t = sp.make_splits(df, FEATS, method='group', seed=sp.SEED)
comp = pd.DataFrame(composition(S_t, 'tree') + composition(S_u, 'union'))
print(comp.to_string(index=False))
big_place = []
for s in sp.SPLIT_SEEDS:
    Ss = sp.make_splits(df_u, FEATS, method='group', seed=s)
    where = {p: int(in_big.loc[Ss[k]].sum()) for p, k in [('train', 'idx_train'), ('val', 'idx_val'), ('test', 'idx_test')]}
    big_place.append(dict(seed=s, **{f'BIG_rows_in_{p}': v for p, v in where.items()}))
big_place = pd.DataFrame(big_place)
print(big_place.to_string(index=False))
composition_diag = dict(big_components=sorted(BIG), big_rows=int(in_big.sum()), big_sybils=int((in_big & syb).sum()),
                        by_partition=comp.to_dict('records'), big_placement_by_seed=big_place.to_dict('records'))

BIG: 2 components, 14,300 rows, 9,214 Sybils (50.6% of Sybils)


split  partition   rows  sybils  sybils_in_BIG  reports  sybils_IxL  sybils_IxI  sybils_IxE  sybils_No provider
 tree      Train 213046    8924           4478      271        7811         783         320                  10
 tree Validation  91305    3824           1968      229        3340         357         120                   7
 tree       Test 130435    5463           2768      239        4770         426         258                   9
union      Train 213046    8924           8260      146        6962        1464         474                  24
union Validation  91305    3824            954       66        3670          35         118                   1
union       Test 130435    5463              0       99        5289          67         106                   1


 seed  BIG_rows_in_train  BIG_rows_in_val  BIG_rows_in_test
   42              13346              954                 0
    1              13346              954                 0
    2              13346              954                 0
    3              13346              954                 0
    4              13346              954                 0
    5              13346              954                 0
    6              13346              954                 0
    7              13346              954                 0
    8              13346              954                 0
    9              13346              954                 0


## Part C. Models on the union split

Hyperparameters from `02` (chosen on the tree split; not re-searched).

In [7]:
def scores(y, p, thr):
    y = np.asarray(y)
    return dict(n=len(y), sybils=int(y.sum()), f1=f1_score(y, (p >= thr).astype(int)),
                ap=average_precision_score(y, p), auroc=roc_auc_score(y, p))

models, partC = [], None
if RUN:
    xgb_p, lgbm_p = sp.load_selected_params()
    leak_u = sp.leakage_report(df, S_u)
    cat_u = df.loc[S_u['idx_test'], 'category']
    xm, lm = sp.fit_xgb(S_u, xgb_p), sp.fit_lgbm(S_u, lgbm_p)
    partC = {}
    for key, nb, m in [('xgb', '03_xgboost_sybil', xm), ('lgbm', '04_lightgbm_sybil', lm)]:
        r = sp.evaluate(f"{sp.load_results(nb)['models'][0]['name']} — union split", S_u['y_val'], m['val'], S_u['y_test'], m['test'])
        sp.print_metrics(r)
        bc = sp.metrics_by_category(S_u['y_test'], m['test'], cat_u, r['threshold'])
        print(bc.round(4).to_string(index=False))
        ref = sp.load_results(nb)['models'][0]
        out_big = ~in_big.loc[S_u['idx_test']].to_numpy()
        partC[key] = dict(rounds=m['rounds'], by_category=bc.to_dict('records'),
                          tree_split_committed=dict(f1=ref['f1'], ap=ref['ap'], code_commit=sp.load_results(nb)['code_commit'],
                                                    note='committed result from another machine; same-machine comparison is Part D'),
                          delta_f1_vs_committed=r['f1'] - ref['f1'], delta_ap_vs_committed=r['ap'] - ref['ap'],
                          test_outside_BIG=scores(S_u['y_test'][out_big], m['test'][out_big], r['threshold']))
        print(f"  vs committed {nb} (other machine): ΔF1 {partC[key]['delta_f1_vs_committed']:+.4f}, "
              f"ΔAP {partC[key]['delta_ap_vs_committed']:+.4f}\n")
        models.append(r)
    sp.save_predictions(df_u, S_u, '15_split_tree_report', prob_xgb_union=xm, prob_lgbm_union=lm)
else:
    print('Skipped (status:', STATUS + ')')

Split method: group
Row overlap (train/val, train/test, val/test): 0, 0, 0
Test addresses sharing a gas provision tree with a train address: 0 / 130,435
Test Sybils sharing a gas provision tree with a train Sybil:     0 / 5,463


  XGBoost seed=42  rounds=575  21.9s


  XGBoost seed=123  rounds=553  34.6s


  XGBoost seed=456  rounds=554  20.9s


  LightGBM seed=42  rounds=370  17.1s


...

  Threshold : 0.5494
  Precision : 0.3947
  Recall    : 0.2209
  F1        : 0.2833
  AUROC     : 0.8110
  Avg Prec  : 0.2254
  Brier     : 0.0418


   category     n  sybil  precision  recall     f1  auroc    fpr
        IxE 27929    106     0.1111  0.0472 0.0662 0.8541 0.0014
        IxI  9771     67     0.0476  0.0448 0.0462 0.8364 0.0062
        IxL 92529   5289     0.4066  0.2267 0.2911 0.7833 0.0201
No provider   206      1     0.0000  0.0000 0.0000 0.9024 0.0049


  vs committed 04_lightgbm_sybil (other machine): ΔF1 -0.4355, ΔAP -0.5442



Saved output/pred_15_split_tree_report_val.parquet (91,305 rows)


Saved output/pred_15_split_tree_report_test.parquet (130,435 rows)


## Part D. 10-split comparison

LightGBM (model seed 42) on the union split and on the tree split for each seed in `sp.SPLIT_SEEDS`, both trained
here, so the differences are paired and on one machine. The overall metrics are computed as in `sp.lgbm_fit_eval`
(asserted equal on the first split). **Same composition**: each model is also scored on its test rows outside
`BIG`, at its own validation threshold.

In [8]:
def lgbm_eval_by_category(S, df_s, params):
    """sp.lgbm_fit_eval's metrics plus test F1 per taxonomy category and the scores outside BIG."""
    r = sp.fit_lgbm(S, params, seeds=(42,), feats=FEATS, verbose=False)
    pv, pt = r['val'], r['test']
    thr = sp.best_f1_threshold(S['y_val'], pv)
    out = dict(threshold=thr)
    for part, y, p in [('val', S['y_val'], pv), ('test', S['y_test'], pt)]:
        out[f'{part}_f1'] = f1_score(y, (p >= thr).astype(int))
        out[f'{part}_ap'] = average_precision_score(y, p)
        out[f'{part}_auroc'] = roc_auc_score(y, p)
    bc = sp.metrics_by_category(S['y_test'], pt, df_s.loc[S['idx_test'], 'category'], thr).set_index('category')
    out.update({f'test_f1_{c}': float(bc.loc[c, 'f1']) for c in ['IxI', 'IxL', 'IxE']})
    ob = ~in_big.loc[S['idx_test']].to_numpy()
    sc = scores(S['y_test'][ob], pt[ob], thr)
    out.update({f'outBIG_{k}': v for k, v in sc.items()})
    return out

per_split, summary_D = None, None
if RUN:
    rows = []
    for s in sp.SPLIT_SEEDS:
        t0 = time.time()
        Su = sp.make_splits(df_u, FEATS, method='group', seed=s)
        St = sp.make_splits(df, FEATS, method='group', seed=s)
        u, t = lgbm_eval_by_category(Su, df_u, lgbm_p), lgbm_eval_by_category(St, df, lgbm_p)
        if s == sp.SPLIT_SEEDS[0]:
            ref = sp.lgbm_fit_eval(Su, FEATS, lgbm_p)
            assert all(abs(ref[k] - u[k]) < 1e-12 for k in ref)
        rows.append(dict(split_seed=s, **{f'union_{k}': v for k, v in u.items()}, **{f'tree_{k}': v for k, v in t.items()}))
        print(f"split {s}: union test F1 {u['test_f1']:.4f}, tree {t['test_f1']:.4f}  ({time.time() - t0:.0f}s)", flush=True)
    per_split = pd.DataFrame(rows)
    assert len(per_split) == 10 and per_split['split_seed'].is_unique
    for m in ['test_f1', 'test_ap', 'test_auroc', 'test_f1_IxI', 'test_f1_IxL', 'test_f1_IxE',
              'outBIG_f1', 'outBIG_ap', 'outBIG_auroc']:
        per_split[f'delta_{m}'] = per_split[f'union_{m}'] - per_split[f'tree_{m}']
    s08 = sp.load_results('08_ablation_families')['summary']['All features']
    num = [c for c in per_split.columns if c != 'split_seed']
    summary_D = {f'{c}_{a}': float(getattr(per_split[c], a)()) for c in num for a in ('mean', 'std')}
    summary_D['union_test_f1_minus_08_all_features_mean'] = summary_D['union_test_f1_mean'] - s08['test_f1_mean']
    show = ['split_seed', 'union_test_f1', 'tree_test_f1', 'delta_test_f1', 'union_outBIG_f1', 'tree_outBIG_f1',
            'delta_outBIG_f1', 'tree_outBIG_sybils', 'union_outBIG_sybils', 'delta_test_f1_IxI', 'delta_test_f1_IxL',
            'delta_test_f1_IxE']
    print(per_split[show].round(4).to_string(index=False))
    print('\nMean ± SD over 10 splits (union − tree, same machine):')
    for m in ['test_f1', 'test_ap', 'test_auroc', 'test_f1_IxI', 'test_f1_IxL', 'test_f1_IxE', 'outBIG_f1', 'outBIG_ap', 'outBIG_auroc']:
        d = per_split[f'delta_{m}']
        print(f"  {m:<14} {summary_D[f'delta_{m}_mean']:+.4f} ± {summary_D[f'delta_{m}_std']:.4f}  (union lower on {(d < 0).sum()} of 10)")
    interpretation_inputs.update(
        union_test_f1_mean=summary_D['union_test_f1_mean'], union_test_f1_sd=summary_D['union_test_f1_std'],
        tree_test_f1_mean=summary_D['tree_test_f1_mean'], tree_test_f1_sd=summary_D['tree_test_f1_std'],
        delta_vs_08_all_features_other_machine=summary_D['union_test_f1_minus_08_all_features_mean'], sd_08=s08['test_f1_std'],
        **{f'delta_{m}_{a}': summary_D[f'delta_{m}_{a}'] for m in ['test_f1', 'test_ap', 'test_f1_IxI', 'test_f1_IxL', 'test_f1_IxE',
                                                                  'outBIG_f1', 'outBIG_ap'] for a in ('mean', 'std')})
else:
    print('Skipped (status:', STATUS + ')')

split 42: union test F1 0.2696, tree 0.7105  (89s)


split 1: union test F1 0.2429, tree 0.6990  (71s)


split 2: union test F1 0.2450, tree 0.7156  (72s)


split 3: union test F1 0.2310, tree 0.7156  (71s)


split 4: union test F1 0.2475, tree 0.7169  (67s)


...

split 6: union test F1 0.2808, tree 0.7105  (66s)


split 7: union test F1 0.2517, tree 0.7194  (73s)


split 8: union test F1 0.2706, tree 0.7175  (71s)


split 9: union test F1 0.2725, tree 0.7093  (65s)


 split_seed  union_test_f1  tree_test_f1  delta_test_f1  union_outBIG_f1  tree_outBIG_f1  delta_outBIG_f1  tree_outBIG_sybils  union_outBIG_sybils  delta_test_f1_IxI  delta_test_f1_IxL  delta_test_f1_IxE
         42         0.2696        0.7105        -0.4409           0.2696          0.6151          -0.3455                2695                 5463            -0.4447            -0.4617            -0.2407
          1         0.2429        0.6990        -0.4561           0.2429          0.6317          -0.3888                2779                 5463            -0.2641            -0.4805            -0.2706
          2         0.2450        0.7156        -0.4705           0.2450          0.6251          -0.3801                2747                 5463            -0.5276            -0.4877            -0.1257
          3         0.2310        0.7156        -0.4846           0.2310          0.6285          -0.3975                2733                 5463            -0.5481            -0.5005

## Save results

In [9]:
sp.save_results(models, '15_split_tree_report', leakage=sp.leakage_report(df, S_u, verbose=False), extra=dict(
    status=STATUS,
    stop_rule='stop before training if a partition Sybil rate deviates > 0.3 pp from the overall rate or a partition '
              'size deviates > 1 pt from 49/21/30; warn if any component holds > 5% of Sybils (a stop in the first run)',
    composition=composition_diag, platform=PLATFORM,
    report_overlap=dict(per_split=overlap.to_dict('records'), summary=overlap_summary),
    components=diag, components_reporter_variant=diag_reporter,
    hyperparameters='from 02_hyperparameter_search (chosen on the tree split; not re-searched)',
    partC=partC, per_split=None if per_split is None else per_split.to_dict('records'), summary=summary_D,
    interpretation_inputs=interpretation_inputs))

Saved results/15_split_tree_report.json
